# Nasdaq-100: sliding-window RP computation benchmark

**Data:** Nasdaq-100 index (`^NDX`), 5-minute regular-session observations, calendar year **2025**. **Window sizes:** 25, 50, 100, 250, 500, 1,000, 2,500. Compare Euclidean with symmetric β-divergence for **β = 1, 3, 7**. Every eligible overlapping window is processed, moving forward one observation at a time. Results report average **seconds per thresholded RP**, not total time per window size.

## Data prerequisite — read before running

[yfinance documents an approximately 60-day limit on intraday history](https://ranaroussi.github.io/yfinance/reference/yfinance.price_history.html). A fresh Yahoo request cannot now retrieve the full 2025 5-minute history. Splitting the year into smaller requests does not recover expired data. This notebook requests exactly 2025 and fails explicitly if unavailable; it never substitutes recent data, daily data, ETFs, or synthetic observations.

If you already archived the original Yahoo 2025 data, set `ARCHIVE_CSV` below to that file. The archive must contain `Datetime` and `Close` columns for **^NDX**, with timezone-aware ISO timestamps (UTC `Z` or numeric offsets). This is an explicit alternative input, not an automatic download fallback. A complete 2025 archive is the missing prerequisite if Yahoo rejects the request.

**No download or benchmark has been executed in this notebook.** Requires NumPy, pandas, Matplotlib only if you later add figures, and yfinance; no RP rendering is performed during benchmarking.

In [3]:
from pathlib import Path
from time import perf_counter
import platform
import sys

import numpy as np
import pandas as pd
import yfinance as yf

TICKER = "^NDX"
START, END = "2025-01-01", "2026-01-01"  # end exclusive
INTERVAL = "5m"
WINDOW_SIZES = (25, 50, 100, 250, 500, 1000, 2500)
METRICS = ("Euclidean", "beta=1", "beta=3", "beta=7")
RR_TARGET = 0.10
ARCHIVE_CSV = None  # e.g. Path("data/NDX_2025_5m.csv"); no file is supplied here

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| yfinance:", yf.__version__)

Python: 3.13.9 | Platform: Windows-11-10.0.26200-SP0
NumPy: 2.4.6 | pandas: 2.3.3 | yfinance: 1.7.0


## 1. Load the exact requested data

The default is yfinance. Set `ARCHIVE_CSV` only for an existing Yahoo archive. Prices are filtered to 2025 and regular-session timestamps in New York. No prices are imputed: invented returns would alter the recurrence workload. Invalid closes and duplicate timestamps are counted and removed. Missing months, non-5-minute inputs, or insufficient observations stop the notebook.

In [4]:
if ARCHIVE_CSV is None:
    try:
        history = yf.Ticker(TICKER).history(
            start=START, end=END, interval=INTERVAL,
            prepost=False, auto_adjust=False, actions=False,
            repair=False, keepna=True, raise_errors=True,
        )
    except Exception as exc:
        raise RuntimeError(
            "Yahoo could not supply ^NDX 5-minute data for all of 2025. "
            "Intraday retention is about 60 days. Set ARCHIVE_CSV to an existing "
            "Yahoo 2025 5-minute archive; this notebook will not substitute other data."
        ) from exc
    source = "yfinance live request"
else:
    archive = pd.read_csv(Path(ARCHIVE_CSV), usecols=["Datetime", "Close"])
    stamps = archive["Datetime"].astype(str)
    if not stamps.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True).all():
        raise ValueError("Archive Datetime values must include UTC Z or numeric timezone offsets")
    history = pd.DataFrame(
        {"Close": pd.to_numeric(archive["Close"], errors="coerce").to_numpy()},
        index=pd.DatetimeIndex(pd.to_datetime(stamps, utc=True)),
    )
    source = f"Saved Yahoo archive: {Path(ARCHIVE_CSV)}"

if history.empty or "Close" not in history:
    raise RuntimeError(
        "No 2025 ^NDX 5-minute closes returned. Yahoo's expired intraday data "
        "requires a previously saved archive; set ARCHIVE_CSV explicitly."
    )
if not isinstance(history.index, pd.DatetimeIndex) or history.index.tz is None:
    raise ValueError("Input must have timezone-aware intraday timestamps")

close = pd.to_numeric(history["Close"], errors="coerce").astype(float)
close.index = close.index.tz_convert("America/New_York")
close = close.sort_index()
duplicate_count = int(close.index.duplicated(keep="last").sum())
close = close.loc[~close.index.duplicated(keep="last")]
close = close.loc[(close.index >= pd.Timestamp(START, tz="America/New_York")) &
                  (close.index < pd.Timestamp(END, tz="America/New_York"))]
close = close.between_time("09:30", "16:00", inclusive="left")
invalid_count = int((~np.isfinite(close) | (close <= 0)).sum())
close = close.loc[np.isfinite(close) & (close > 0)].rename("NDX_CLOSE")
if close.empty:
    raise ValueError("No valid 2025 regular-session closes remain")
if np.any(close.index.minute % 5 != 0) or np.any(close.index.second != 0):
    raise ValueError("Timestamps are not on a five-minute grid")
months = close.index.tz_localize(None).to_period("M").unique()
expected_months = pd.period_range("2025-01", "2025-12", freq="M")
if not months.equals(expected_months):
    raise ValueError(f"Expected all 12 months of 2025; received {list(months)}")
steps = close.index.to_series().diff().dropna()
if steps.empty or steps.mode().iloc[0] != pd.Timedelta(minutes=5):
    raise ValueError("Input is not predominantly five-minute observations")

print(source)
print(f"Clean closes: {len(close):,}; {close.index[0]} to {close.index[-1]}")
print(f"Duplicate timestamps removed: {duplicate_count}; invalid closes removed: {invalid_count}")
print("Observed valid closes per month (review for archive gaps):")
print(close.groupby(close.index.tz_localize(None).to_period("M")).size().to_string())

RuntimeError: Yahoo could not supply ^NDX 5-minute data for all of 2025. Intraday retention is about 60 days. Set ARCHIVE_CSV to an existing Yahoo 2025 5-minute archive; this notebook will not substitute other data.

## 2. Prepare comparable positive inputs

Use **gross five-minute returns** $g_t=P_t/P_{t-1}=\exp(r_t)$, where $r_t$ is the decimal log return. All four methods use the **same** input values and scalar states (embedding dimension 1), isolating divergence-computation cost rather than mixing representations. This differs from the z-scored Euclidean inputs in the earlier analysis notebooks.

Retain a return only when its two prices are exactly five minutes apart. Overnight, weekend, and missing-bar returns are excluded. Windows contain consecutive **retained observations**, so they can cross session breaks; their length is an observation count, not uninterrupted wall-clock minutes. Incomplete windows are not used.

In [ ]:
five_minute_pair = close.index.to_series().diff().eq(pd.Timedelta(minutes=5))
log_returns = np.log(close).diff().loc[five_minute_pair]
gross_returns = np.exp(log_returns)
values = np.ascontiguousarray(gross_returns.to_numpy(dtype=np.float64))
if not np.isfinite(values).all() or np.any(values <= 0):
    raise ValueError("Gross returns must be finite and strictly positive")
if len(values) < max(WINDOW_SIZES):
    raise ValueError(f"Need at least {max(WINDOW_SIZES)} valid returns; got {len(values)}")

workload = pd.DataFrame({
    "window_size": WINDOW_SIZES,
    "windows": [len(values) - size + 1 for size in WINDOW_SIZES],
})
print(f"Retained five-minute returns: {len(values):,}")
print(f"Excluded non-five-minute transitions: {len(close) - 1 - len(values):,}")
print(workload.to_string(index=False))
print(f"Total thresholded RPs to compute: {len(METRICS) * int(workload['windows'].sum()):,}")

## 3. What is timed?

Each call constructs the **full square divergence/distance matrix**, selects ε as the 10th percentile of off-diagonal upper-triangle entries, and produces the full Boolean thresholded RP. Matrix allocation, per-window β transforms, quantile selection, and thresholding are included. Data fetching/cleaning, return preparation, window slicing, shared upper-triangle index preparation, warm-up calls, reporting, and Matplotlib rendering are excluded.

For scalar states, Euclidean distance is $|x-y|$. As in the previous notebooks, use the symmetric mean of forward and reverse β-divergences:

$$
S_1(x,y)=\tfrac12(x-y)(\log x-\log y),\qquad
S_\beta(x,y)=\frac{(x-y)(x^{\beta-1}-y^{\beta-1})}{2(\beta-1)}.
$$

[β-divergence definition: Févotte & Idier, §2.1](https://arxiv.org/abs/1010.1763). Thresholds are fitted separately for each method and window; they have different units. Ties may make actual recurrence exceed the target. Symmetrization is included for β methods. Only one returned RP is kept at a time; no collection of window matrices is stored.

In [ ]:
def build_thresholded_rp(window, metric, upper, target_rr):
    distance = np.subtract.outer(window, window)
    if metric == "Euclidean":
        np.abs(distance, out=distance)
    else:
        beta = int(metric.split("=")[1])
        transformed = np.log(window) if beta == 1 else window ** (beta - 1) / (beta - 1)
        distance *= np.subtract.outer(transformed, transformed)
        distance *= 0.5
    # Advanced indexing creates a temporary vector; quantile may reorder that copy.
    epsilon = np.quantile(distance[upper], target_rr, overwrite_input=True)
    return distance <= epsilon

## 4. Run all sliding windows — computationally intensive

**This cell performs the benchmark.** For every requested size, move forward by one retained observation and process **every** full window with all four methods. Nothing is subsampled. Each window produces a new RP from scratch; this does not benchmark an incremental-update algorithm.

A small, untimed warm-up precedes the measurements. Method order rotates between windows to reduce systematic first/last-method bias. Each RP is timed once with `perf_counter`; the average is over all windows of that size. Results depend on CPU, NumPy version, memory pressure, and other running processes. Overlapping windows are not independent statistical replicates. Keep the machine otherwise idle.

The largest windows require quadratic memory per matrix. Rendering thousands of images would measure plotting overhead and is deliberately omitted.

In [ ]:
# Small untimed warm-up; no benchmark or download runs before you execute these cells.
warm_size = min(WINDOW_SIZES)
warm_upper = np.triu_indices(warm_size, k=1)
for metric in METRICS:
    warm_rp = build_thresholded_rp(values[:warm_size], metric, warm_upper, RR_TARGET)
    del warm_rp

rows = []
for size in WINDOW_SIZES:
    upper = np.triu_indices(size, k=1)  # shared by every method/window of this size
    n_windows = len(values) - size + 1
    elapsed = {metric: 0.0 for metric in METRICS}
    for start in range(n_windows):
        window = values[start : start + size]  # view; not timed
        shift = start % len(METRICS)
        order = METRICS[shift:] + METRICS[:shift]
        for metric in order:
            t0 = perf_counter()
            rp = build_thresholded_rp(window, metric, upper, RR_TARGET)
            seconds = perf_counter() - t0
            elapsed[metric] += seconds
            del rp  # release before the next call; never retain all window RPs
    for metric in METRICS:
        rows.append({
            "window_size": size,
            "metric": metric,
            "windows": n_windows,
            "total_seconds": elapsed[metric],
            "mean_seconds_per_rp": elapsed[metric] / n_windows,
        })
    print(f"Completed size {size:,}: {n_windows:,} windows for each method")
    del upper

benchmark = pd.DataFrame(rows)
print("Benchmark complete.")

## 5. Average seconds per recurrence plot

Each entry is **total measured construction time / number of sliding windows** for that size and method. Download time, data preparation, and rendering are not included. No timing values are pre-filled. `benchmark` also retains the window counts and measured totals for inspection.

In [ ]:
average_seconds = benchmark.pivot(
    index="window_size", columns="metric", values="mean_seconds_per_rp",
).reindex(index=WINDOW_SIZES, columns=METRICS)
average_seconds.index.name = "RP size (N × N)"
average_seconds.columns.name = "Mean seconds per thresholded RP"
average_seconds.style.format("{:.6f}")